# FastAPI Backend Specialist — QLoRA Fine-Tuning (Colab Pro)

LoRA / QLoRA (4-bit) fine-tuning of a small 1-3B base model on FastAPI
task->code pairs, per the *democratized AI coding* architecture (Phase 1 =
FastAPI backend specialist).

**Runtime:** Colab Pro, GPU runtime (T4/L4/A100). `Runtime -> Change runtime
type -> GPU`.

**This notebook runs end-to-end on DUMMY data** (a handful of verified pairs
embedded below). Point it at the real `fastapi-backend-pairs.jsonl` later by
uploading the file — the loader picks it up automatically.

> Do **not** treat the tiny dummy run as a real training run — it exists only to
> prove the pipeline executes. Real training uses the full verified dataset.

**Fix applied (vs. earlier version):** loss is now masked so the model only
learns to predict the *code* completion, not the task prompt, and padding
tokens no longer contribute to the loss. See section 6 for details.

**Persistence:** this version mounts Google Drive (section 1b) so the dataset and the saved adapter both survive a runtime disconnect — see 1b for one-time setup.

## 1. Install dependencies

In [ ]:
# Colab already ships torch. Pin the fine-tuning stack.
!pip install -q -U "transformers>=4.44" "peft>=0.12" "accelerate>=0.33" \
    "bitsandbytes>=0.43" "datasets>=2.20"

## 1b. Mount Google Drive (persistence)

Colab session storage is wiped whenever the runtime disconnects (idle timeout,
closing the tab, or a natural reset) — this is more frequent on the **free**
tier than Colab Pro. Mounting Drive means the dataset only needs to be
uploaded once, and the trained adapter is saved somewhere that survives a
disconnect right after training finishes (exactly what happened in an earlier
run here).

**One-time setup:** create a folder in your Drive (e.g. `MyDrive/open_web/`)
and upload `fastapi-backend-pairs.jsonl` (and later `fastapi-eval-suite.jsonl`)
there via drive.google.com, or upload directly through the file browser on the
left after mounting below.

**Note:** mounting Drive does NOT preserve in-memory Python state (the loaded
model, tokenized dataset, etc.) across a disconnect — only files. You'll still
need to re-run the model-loading/tokenize/LoRA cells after a reconnect; what
Drive fixes is not needing to re-upload the dataset, and not losing a
finished adapter if save happens to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this to match where you put/will put your project files in Drive.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/open_web"
import os
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print("Drive mounted. Project dir:", DRIVE_PROJECT_DIR)

## 2. Configuration

In [ ]:
import torch

# Small base model (1-3B). Swap for another candidate (e.g. a MiniCPM / Qwen
# coder variant) when comparing base models per the architecture doc.
MODEL_ID   = "Qwen/Qwen2.5-Coder-1.5B"   # ~1.5B, permissive, strong on code

# Paths now point at the mounted Drive folder (see 1b above) so the dataset
# and the trained adapter both persist across session disconnects.
DATA_PATH  = f"{DRIVE_PROJECT_DIR}/fastapi-backend-pairs.jsonl"
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/fastapi-backend-lora"
MAX_LEN    = 1024

# 4-bit QLoRA requires a CUDA GPU. Fall back to a CPU-friendly path if none.
USE_4BIT = torch.cuda.is_available()
print("CUDA available:", torch.cuda.is_available(), "| 4-bit QLoRA:", USE_4BIT)

## 3. Load the dataset

Uses `fastapi-backend-pairs.jsonl` if present (upload it via the Files pane),
otherwise falls back to a small embedded dummy set so the notebook always runs.

In [ ]:
import json, os

DUMMY_PAIRS = [
    {"task": "Create a FastAPI app with a GET /items endpoint that returns all items.",
     "code": "from fastapi import FastAPI\napp = FastAPI()\nitems = [{'id': 1, 'name': 'apple'}]\n\n@app.get('/items')\ndef list_items():\n    return items\n"},
    {"task": "Create a FastAPI GET /items/{item_id} endpoint returning 404 if missing.",
     "code": "from fastapi import FastAPI, HTTPException\napp = FastAPI()\nitems = {1: {'id': 1, 'name': 'apple'}}\n\n@app.get('/items/{item_id}')\ndef get_item(item_id: int):\n    if item_id not in items:\n        raise HTTPException(status_code=404, detail='Item not found')\n    return items[item_id]\n"},
    {"task": "Create a FastAPI POST /items endpoint that returns 201 with an id.",
     "code": "from fastapi import FastAPI, status\nfrom pydantic import BaseModel\napp = FastAPI()\n_db = {}\n\nclass ItemIn(BaseModel):\n    name: str\n\n@app.post('/items', status_code=status.HTTP_201_CREATED)\ndef create_item(item: ItemIn):\n    new = {'id': len(_db) + 1, 'name': item.name}\n    _db[new['id']] = new\n    return new\n"},
]

if os.path.exists(DATA_PATH):
    pairs = [json.loads(l) for l in open(DATA_PATH, encoding='utf-8')]
    print(f"Loaded {len(pairs)} real pairs from {DATA_PATH}")
else:
    pairs = DUMMY_PAIRS
    print(f"No {DATA_PATH} found — using {len(pairs)} embedded DUMMY pairs")

## 4. Prompt formatting

A simple instruction template. The frozen contract from planning (the future
input to this specialist) would replace the free-text task later; the shape
stays task -> code.

**Kept as two separate fields (prompt, completion) rather than one merged
`text` field** — this is what makes prompt-masking in section 6 possible.
The previous version merged them into one `text` string before tokenizing,
which loses the boundary needed to mask the prompt out of the loss.

In [ ]:
PROMPT_PREFIX = "### Task:\n{task}\n\n### FastAPI code:\n"

def format_example(ex):
    prompt = PROMPT_PREFIX.format(task=ex["task"])
    completion = ex["code"]
    return {"prompt": prompt, "completion": completion}

from datasets import Dataset
raw_ds = Dataset.from_list(pairs).map(format_example)
print(raw_ds[0]["prompt"][:200])
print("---")
print(raw_ds[0]["completion"][:200])

## 5. Tokenizer & 4-bit base model (QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto"
    )
else:
    # CPU fallback (no bitsandbytes) so the notebook still runs anywhere.
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)

model.config.use_cache = False

## 6. Tokenize — WITH LABEL MASKING (the fix)

**Problem in the earlier version:** `labels = input_ids.copy()` trained the
model to predict the *entire* sequence, including the `### Task: ...` prompt
text and the padding tokens added to reach `MAX_LEN`. That wastes training
signal on text the model doesn't need to generate, and adds noisy gradient
signal from padding on the (likely majority of) shorter examples.

**Fix:** tokenize the prompt and completion separately, concatenate them,
then set `labels` so that:
- prompt tokens -> `-100` (ignored in the loss — the model isn't trained to
  reproduce the task description)
- completion tokens -> real token ids (this is what the model should learn
  to generate)
- padding tokens -> `-100` (ignored in the loss)

`-100` is the standard PyTorch/HF convention for "ignore this position in
`CrossEntropyLoss`".

In [ ]:
IGNORE_INDEX = -100

def tokenize_with_masking(example):
    prompt_ids = tokenizer(example["prompt"], add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(
        example["completion"], add_special_tokens=False
    )["input_ids"] + [tokenizer.eos_token_id]  # eos marks end of generation

    input_ids = prompt_ids + completion_ids
    labels = ([IGNORE_INDEX] * len(prompt_ids)) + completion_ids

    # Truncate to MAX_LEN (keep from the start; drop tail if too long)
    input_ids = input_ids[:MAX_LEN]
    labels = labels[:MAX_LEN]

    attention_mask = [1] * len(input_ids)

    # Right-pad to MAX_LEN; padding contributes to attention_mask=0 and
    # labels=-100, so it's excluded from both attention and loss.
    pad_len = MAX_LEN - len(input_ids)
    if pad_len > 0:
        input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
        attention_mask = attention_mask + [0] * pad_len
        labels = labels + [IGNORE_INDEX] * pad_len

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

tokenized_ds = raw_ds.map(
    tokenize_with_masking,
    remove_columns=raw_ds.column_names,
)

# Sanity check: confirm prompt + padding are masked, completion isn't.
sample = tokenized_ds[0]
n_masked = sum(1 for l in sample["labels"] if l == IGNORE_INDEX)
n_real = sum(1 for l in sample["labels"] if l != IGNORE_INDEX)
print(f"Example 0 — masked (prompt+pad) tokens: {n_masked}, trainable (completion) tokens: {n_real}")

## 7. Attach LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Train

`num_train_epochs=3` confirmed as a reasonable default on the real 71-pair
dataset (produces a healthy loss curve, ~0.87 -> ~0.07-0.13 over 54 steps).
Raise further or use the full dataset as it grows; iterate on **dataset
quality** first, hyperparameters second.

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=USE_4BIT,
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds,
    data_collator=default_data_collator,
)
trainer.train()

## 9. Save the adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter saved to:", OUTPUT_DIR)
# The LoRA adapter is only tens of MB — download it and run base+adapter locally.

## 10. Quick inference sanity check

**Fix applied:** gradient checkpointing (enabled during training by
`prepare_model_for_kbit_training` in section 7) must be explicitly disabled
before generation, along with `model.eval()`. Leaving it enabled silently
forces `use_cache=False` during `generate()`, breaking KV caching and
producing degenerate/repeating output regardless of how well-trained the
adapter actually is — this was the root cause of the earlier garbage output
(`from 200000...`), not undertraining.

In [ ]:
# Disable gradient checkpointing before generation — prepare_model_for_kbit_training()
# (section 7) enables it by default for memory-efficient training, but it silently
# forces use_cache=False during generation (see the transformers warning), which
# breaks KV caching and produces degenerate/repeating output (e.g. "0000...").
# Must disable it and set eval mode before generating.
model.gradient_checkpointing_disable()
model.eval()
model.config.use_cache = True

prompt = "### Task:\nCreate a FastAPI GET /health endpoint returning {'status':'ok'}.\n\n### FastAPI code:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=128, do_sample=False,
                     pad_token_id=tokenizer.pad_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## Notes for scaling up

- Upload the full `fastapi-backend-pairs.jsonl` (cell 3 auto-detects it).
- Compare base models (Qwen coder / MiniCPM etc.) by re-running with a different
  `MODEL_ID` and scoring each against `fastapi-eval-suite.jsonl` using
  `eval_runner.py`.
- Iterate on **dataset quality** first, hyperparameters second (per the brief).
- Keep `use_cache=False` during training, `True` for generation (handled above).
- **Label masking (section 6) must be preserved** in any future edits to the
  tokenization step — reverting to `labels = input_ids.copy()` (or reintroducing
  `DataCollatorForLanguageModeling`, which recomputes labels the same way)
  silently undoes the fix.